# Reproducible scripts for Hydrogen End-Use: Power, Heat, Mobility, and Industry

This companion notebook was generated from fenced `python` code blocks in `chapters/ch18_h2_end_use/chapter.md`. Blocks marked `noexec` in the chapter are preserved for reading but are not executed here.


In [1]:
from pathlib import Path
import re
import shutil

try:
    from IPython.display import Image, display
except Exception:
    Image = None
    display = None

_CHAPTER_FIGURES_DIR = Path("../figures")
_CHAPTER_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
_PAPERLAB_PATTERNS = ("*.png", "*.jpg", "*.jpeg", "*.svg", "*.webp", "*.csv", "*.json", "*.txt", "*.html")
_PAPERLAB_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg"}


def _paperlab_stamp(path):
    try:
        stat = path.stat()
        return (stat.st_mtime_ns, stat.st_size)
    except OSError:
        return None


def _paperlab_scan_files():
    files = {}
    roots = [Path("."), _CHAPTER_FIGURES_DIR]
    for root in roots:
        if not root.exists():
            continue
        for pattern in _PAPERLAB_PATTERNS:
            for path in root.glob(pattern):
                if path.is_file():
                    files[path.resolve()] = _paperlab_stamp(path)
    return files


def _paperlab_safe_name(name):
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("._")
    return cleaned or "notebook_output"


_paperlab_seen_files = _paperlab_scan_files()


def _paperlab_capture_new_files(label):
    global _paperlab_seen_files
    current = _paperlab_scan_files()
    changed = []
    for path, stamp in current.items():
        if _paperlab_seen_files.get(path) != stamp:
            changed.append(Path(path))
    _paperlab_seen_files = current

    for path in sorted(changed):
        suffix = path.suffix.lower()
        if suffix in _PAPERLAB_IMAGE_SUFFIXES:
            if path.parent.resolve() == _CHAPTER_FIGURES_DIR.resolve():
                dest = path
            else:
                dest = _CHAPTER_FIGURES_DIR / _paperlab_safe_name(f"{label}_{path.name}")
                shutil.copy2(path, dest)
            print(f"Captured figure: figures/{dest.name}")
            if Image is not None and display is not None:
                display(Image(filename=str(dest)))
        else:
            print(f"Generated result file: {path}")

## Example 1 from `chapters/ch18_h2_end_use/chapter.md` line 116


This block is preserved but not executed because it is noexec marker.

````python
from neqsim import jneqsim as J

# Direct Java access through neqsim-python. Use explicit units and call
# setMixingRule before running flashes or process equipment.
# Wobbe Index sweep for H2 blending into a natural-gas grid (ISO 6976 basis).
# H = ISO 6976 calculator; H_inferior is reported per Sm3 dry at 15 degC.
ref_T_C = 15.0
ref_P_C = 15.0
results = []
for h2_vol_frac in [0.0, 0.05, 0.10, 0.20, 0.30, 0.50]:
    gas = J.thermo.system.SystemSrkEos(288.15, 50.0)
    gas.addComponent("hydrogen", h2_vol_frac)
    gas.addComponent("methane", 0.90 * (1.0 - h2_vol_frac))
    gas.addComponent("ethane", 0.06 * (1.0 - h2_vol_frac))
    gas.addComponent("propane", 0.03 * (1.0 - h2_vol_frac))
    gas.addComponent("nitrogen", 0.01 * (1.0 - h2_vol_frac))
    gas.setMixingRule("classic")
    iso = J.standards.gasquality.Standard_ISO6976(gas, ref_T_C, ref_P_C, "volume")
    iso.calculate()
    results.append({
        "h2_vol_frac": h2_vol_frac,
        "Wobbe_MJ_per_Sm3": iso.getValue("SuperiorWobbeIndex") / 1.0e6,
        "GCV_MJ_per_Sm3": iso.getValue("SuperiorCalorificValue") / 1.0e6,
        "rel_density": iso.getValue("RelativeDensity"),
    })

# Fuel-cell baseline efficiency (LHV basis) - illustrative
fuel_cell_eta_LHV = 0.55
for r in results:
    print(r)
print("PEMFC LHV efficiency (typical)", fuel_cell_eta_LHV)

````
